In [1]:
import os
import sys
import warnings
warnings.filterwarnings('ignore')
warnings.simplefilter('ignore')

import xarray as xr
xr.set_options(keep_attrs=True)
import numpy as np
np.seterr(divide='ignore', invalid='ignore')
from scipy.stats import ttest_ind

import cartopy
cartopy.config['data_dir'] = "/discover/nobackup/projects/jh_tutorials/JH_examples/JH_datafiles/Cartopy"
cartopy.config['pre_existing_data_dir'] = "/discover/nobackup/projects/jh_tutorials/JH_examples/JH_datafiles/Cartopy"
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import matplotlib.patches as patches
import matplotlib.colors as mcolors
from matplotlib import cm
import matplotlib.patheffects as path_effects

# settings
%config InlineBackend.figure_format = 'retina'

# add path to custom functions
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path+"/py_functions")
from map_plot_tools import *
from colorbar_funcs import *
from data_funcs import *
from stats_funcs import *
from domain_funcs import *

dpath0 = '/discover/nobackup/projects/giss/baldwin_nip/dmkumar'

In [2]:
def build_diff_dataset(results, metric, domains, models):
    """Build an xr.Dataset of lr, hr, and hr-lr difference values per domain."""
    ds = xr.Dataset()
    for domain in domains:
        lr_vals = [results['lr'][domain][m][metric] for m in models]
        hr_vals = [results['hr'][domain][m][metric] for m in models]
        lr_da = xr.DataArray(np.array(lr_vals), coords={"model": models}, dims=["model"])
        hr_da = xr.DataArray(np.array(hr_vals), coords={"model": models}, dims=["model"])
        ds[f"{metric}_lr_{domain}"]   = lr_da
        ds[f"{metric}_hr_{domain}"]   = hr_da
        ds[f"{metric}_diff_{domain}"] = hr_da - lr_da
    return ds

In [3]:
models=['CAM-MPAS',
        'CMCC-CM2',
        #'CNRM-CM6-1',
        'EC-Earth3P',
        'FGOALS-f3',
        'HadGEM3-GC31',
        'HiRAM-SIT',
        'MPI-ESM1-2']#,
        #'MRI-AGCM3-2']

hrkeys=['CAM-MPAS-HR',
        'CMCC-CM2-VHR4',
        #'CNRM-CM6-1-HR',
        'EC-Earth3P-HR',
        'FGOALS-f3-H',
        'HadGEM3-GC31-HM',
        'HiRAM-SIT-HR',
        'MPI-ESM1-2-XR']#,
        #'MRI-AGCM3-2-S']

lrkeys=['CAM-MPAS-LR',
        'CMCC-CM2-HR4',
        #'CNRM-CM6-1',
        'EC-Earth3P',
        'FGOALS-f3-L',
        'HadGEM3-GC31-LM',
        'HiRAM-SIT-LR',
        'MPI-ESM1-2-HR']#,
        #'MRI-AGCM3-2-H']

hr_map = dict(zip(models, hrkeys))
lr_map = dict(zip(models, lrkeys))

key_map = {'hr': hr_map,
           'lr': lr_map}

resolutions = ['lr', 'hr']
experiments = ['highresSST-present', 'highresSST-future']

In [4]:
coords_map, regions = nam_regions()
domains = list(coords_map.keys())

In [5]:
# --- define time intervals once --- #
fut_st_yr,  fut_end_yr = '2030', '2050'
hist_st_yr,  hist_end_yr = '1950', '2014'

## If running for the first time, execute these steps
Loads and processes HRMIP highresSST-future data

In [ ]:
## +++ LOAD & ORGANIZE DATA +++ ##

varn = 'pr'
latmin, latmax = 10, 70
lonmin, lonmax = 200, 300

#=== HighResMIP models ===
# data paths
files = {}
for case in experiments:
    files[case] = {}
    for model in models:
        files[case][model] = {
            'lr': f'{dpath0}/hrmip/{case}/{varn}/{varn}.Amon.{case}.{lr_map[model]}.nc',
            'hr': f'{dpath0}/hrmip/{case}/{varn}/{varn}.Amon.{case}.{hr_map[model]}.nc',
        }

# load model data
dat_full = {res: {exp: {} for exp in experiments} for res in resolutions}
dat      = {res: {exp: {} for exp in experiments} for res in resolutions}

print('Loading model data for...')
for res in resolutions:
    print(f"\n{res}")
    
    for case in experiments:
        print(f"  -> {case}")
        
        for model in models:
            print(f"     ---> {model}")
            ds = xr.open_dataset(files[case][model][res] , chunks={'time':12})[varn]
            ds = ds.sortby('time')
            ds = match_lat_lon_names(ds) * 86400  # mm/s → mm/day
            if case == 'highresSST-future':
                ds = ds.sel(time=slice(fut_st_yr, fut_end_yr))
            else:
                ds = ds.sel(time=slice(hist_st_yr, hist_end_yr))
            ds = ds.squeeze()
            ds = ds.drop_vars([c for c in ['member_id', 'dcpp_init_year'] if c in ds.coords])
            dat_full[res][case][model] = ds
            dat[res][case][model]      = ds.sel(lat=slice(latmin, latmax), lon=slice(lonmin, lonmax))

# store information about model resolution ===
cfg_resolution = {}

print('\nGrabbing model resolution info...')
for res in resolutions:
    for model in models:
        cfg = lr_map[model] if res == 'lr' else hr_map[model]
        cfg_resolution[cfg] = len(dat_full[res]['highresSST-present'][model].lon)
        
# calculate seasonal means
print('\nCalculating model seasonal means...')
jas_mean     = { res: { exp: { model: {} for model in models } for exp in experiments } for res in resolutions }
ann_jas_mean = { res: { exp: { model: {} for model in models } for exp in experiments } for res in resolutions }

for res in resolutions:
    for case in experiments:
        for model in models:
            jas_mean[res][case][model] = jas_seasonal_mean(dat[res][case][model]).load()
            ann_jas_mean[res][case][model] = jas_yearly_mean(dat[res][case][model]).load()


#=== Obs ===
# topo
etopo_full = xr.open_dataset(f'{dpath0}/topo_files/obs.etopo5.zsurf.nc').ROSE.rename({'ETOPO05_X':'lon', 'ETOPO05_Y':'lat'})
etopoNA = etopo_full.where(etopo_full>0, np.nan).sel(lat=slice(latmin,latmax), lon=slice(lonmin,lonmax))

# precip
obs_dat = {}
obs_jas = {}
obs_ann_jas = {}

print('\nLoading obs data and calculating seasonal means...')
"""
# TRMM
obs_dat['trmm']     = xr.open_dataset(f'{dpath0}/obs_data/prec/pr_TRMM-L3_v7-7A_199801-201312.nc', chunks={'time':12}).pr.sel(lat=slice(latmin,latmax), lon=slice(lonmin,lonmax)) * 86400 # kg m-2 s-1 -> mm/day
obs_jas['trmm']     = jas_seasonal_mean(obs_dat['trmm'])
obs_ann_jas['trmm'] = jas_yearly_mean(obs_dat['trmm'])
"""
# IMERG
imerg = xr.open_dataset(f'{dpath0}/obs_data/prec/imerg.gn.timeseries.2001-2018.nc' , chunks={'time':12}).precipitation * 24 # mm/hr -> mm/day
imerg = imerg.transpose('time','lat','lon')
imerg.attrs['units'] = 'mm/day'
imerg.attrs['Units'] = 'mm/day'
imerg = lonFlip(imerg)
obs_dat['imerg']     = imerg.sel(lat=slice(latmin,latmax), lon=slice(lonmin,lonmax))
obs_jas['imerg']     = jas_seasonal_mean(obs_dat['imerg']).load()
obs_ann_jas['imerg'] = jas_yearly_mean(obs_dat['imerg']).load()
del imerg

# regrid to model resolutions
print('\nRegridding obs to model resolutions...')
obs_regrid     = { res: { model: {} for model in models } for res in resolutions } 
obs_ann_regrid = { res: { model: {} for model in models } for res in resolutions } 

for res in resolutions:
    for model, da in dat[res]['highresSST-present'].items():
        obs_regrid[res][model]     = obs_jas['imerg'].interp(lat=da.lat, lon=da.lon)
        obs_ann_regrid[res][model] = obs_ann_jas['imerg'].interp(lat=da.lat, lon=da.lon)


print('\nDone.')

Loading model data for...

lr
  -> highresSST-present
     ---> CAM-MPAS
     ---> CMCC-CM2
     ---> EC-Earth3P
     ---> FGOALS-f3
     ---> HadGEM3-GC31
     ---> HiRAM-SIT
     ---> MPI-ESM1-2
  -> highresSST-future
     ---> CAM-MPAS
     ---> CMCC-CM2
     ---> EC-Earth3P
     ---> FGOALS-f3
     ---> HadGEM3-GC31
     ---> HiRAM-SIT
     ---> MPI-ESM1-2

hr
  -> highresSST-present
     ---> CAM-MPAS
     ---> CMCC-CM2
     ---> EC-Earth3P
     ---> FGOALS-f3
     ---> HadGEM3-GC31
     ---> HiRAM-SIT
     ---> MPI-ESM1-2
  -> highresSST-future
     ---> CAM-MPAS
     ---> CMCC-CM2
     ---> EC-Earth3P
     ---> FGOALS-f3
     ---> HadGEM3-GC31
     ---> HiRAM-SIT
     ---> MPI-ESM1-2

Grabbing model resolution info...

Calculating model seasonal means...


In [ ]:
## +++ COMPARING ALL MODEL FUTURE TO MODEL PI +++ ##

future_diff      = { res: { model: {} for model in models } for res in resolutions } 
future_diff_mask = { res: { model: {} for model in models } for res in resolutions }
future_ptvals    = { res: { model: {} for model in models } for res in resolutions }

# calculate significance of future - present difference
print('Significance testing for:')

for res in resolutions:
    print(f'{res}')
    
    for model in models:
        print(f'-> {model}')
            
        diff_, diff_mask_, ptvals_ = sigtest2n(ann_jas_mean[res]['highresSST-future'][model], ann_jas_mean[res]['highresSST-present'][model],
                                               jas_mean[res]['highresSST-future'][model], jas_mean[res]['highresSST-present'][model])
        future_diff[res][model] = diff_
        future_diff_mask[res][model] = diff_mask_
        future_ptvals[res][model] = ptvals_

        m = future_diff_mask[res][model]
        future_diff_mask[res][model] = future_diff[res][model].where(~np.ma.getmaskarray(m))

print('Done.')

In [ ]:
# plot to see how the northern NAM domain bounds lie w.r.t. SMO & North American topography
cmap,_,_,_=get_settings(field='precip', diff=False)
vmin=0
vmax=9
levels=np.linspace(vmin, vmax, 19)
norm=mpl.colors.BoundaryNorm(levels, cmap.N)
proj, trans = ccrs.PlateCarree(), ccrs.PlateCarree()

fig = plt.figure(figsize=(5, 6))
ax = plt.axes(projection=proj)

# obs jas precip
cf = ax.pcolormesh(obs_jas['imerg'].lon, obs_jas['imerg'].lat, obs_jas['imerg'], cmap=cmap, norm=norm, transform=trans)
# etopo05 contours
ax.contour(etopoNA.lon, etopoNA.lat, etopoNA, levels=np.linspace(800,3800,7), linewidths=0.5, colors='k', transform=trans)
# add box around NAM sub-domains
poly1=patches.Polygon(coords_map['north'], closed=True, ec='firebrick', fc='none', lw=1.5, ls='--', transform=ccrs.PlateCarree(), zorder=100); ax.add_patch(poly1)
poly2=patches.Polygon(coords_map['south'], closed=True, ec='red', fc='none', lw=1.5, ls='--', transform=ccrs.PlateCarree(), zorder=100); ax.add_patch(poly2)
t1=ax.text(-114, 34.75,'NORTHERN', color='firebrick', fontsize=11, fontweight='bold')
t2=ax.text(-112, 20.25,'SOUTHERN', color='red', fontsize=11, fontweight='bold')
t1.set_path_effects([path_effects.Stroke(linewidth=1.5, foreground='white'),path_effects.Normal()])
t2.set_path_effects([path_effects.Stroke(linewidth=1.5, foreground='white'),path_effects.Normal()])

# map formatting
ax.coastlines(linewidth=1.25)
ax.add_feature(cfeature.BORDERS, edgecolor='k', linewidth=1)
ax.add_feature(cfeature.STATES, edgecolor='k', linewidth=1)
ax.set_extent([239,260,15,40], crs=proj)
gl = ax.gridlines(draw_labels=True, linewidth=0.5, color='gray', alpha=0.75, linestyle='--')
gl.top_labels = False; gl.right_labels = False
gl.xlocator = plt.FixedLocator(range(-180, 181, 5))
gl.ylocator = plt.FixedLocator(range(-90, 91, 5))

# colorbar
cax=fig.add_axes([.925, 0.15, 0.035, 0.7])
cbar=fig.colorbar(cf, ticks=levels, orientation='vertical', extend='max', cax=cax)
cbar.set_label('IMERG Precipitation Rate [mm day$^{-1}$]', labelpad=16, rotation=270, size=10, fontweight='normal', ha='center')
cbar.ax.tick_params(labelsize=10)
for tick in cbar.ax.yaxis.get_major_ticks():
    tick.label2.set_fontweight('normal')

In [ ]:
## +++ Calculate area-weighted mean pr for southern and northern NAM domains +++ ##

results = {res: {d: {} for d in domains} for res in resolutions}

for res in resolutions:
    for domain in domains:
        for model, hist_pr in jas_mean[res]['highresSST-present'].items():
            imerg    = obs_regrid[res][model]
            fut_diff = future_diff[res][model]

            hist_climo    = regional_weighted_mean(hist_pr, regions).sel(region=domain).load()
            hist_bias     = regional_weighted_mean(hist_pr - imerg, regions).sel(region=domain).load()
            hist_bias_abs = np.abs(hist_bias)
            pct_change    = (fut_diff / hist_pr) * 100
            pct_mean      = regional_weighted_mean(pct_change, regions).sel(region=domain).load()

            # Welch's two-sample t-test: is the regional-mean future change significant?
            ann_pres    = regional_weighted_mean(ann_jas_mean[res]['highresSST-present'][model], regions).sel(region=domain).values
            ann_fut     = regional_weighted_mean(ann_jas_mean[res]['highresSST-future'][model], regions).sel(region=domain).values
            _, pct_pval = ttest_ind(ann_fut, ann_pres, equal_var=False)

            # Welch's two-sample t-test: is the historical bias difference significant?
            ann_hist         = regional_weighted_mean(ann_jas_mean[res]['highresSST-present'][model], regions).sel(region=domain).values
            ann_hist_obs     = regional_weighted_mean(obs_ann_jas['imerg'], regions).sel(region=domain).values
            _, hist_pval_obs = ttest_ind(ann_hist, ann_hist_obs, equal_var=False)
            
            results[res][domain][model] = {
                "hist_climo"    : hist_climo,
                "hist_bias"     : hist_bias,
                "hist_bias_abs" : hist_bias_abs,
                "future_pct"    : pct_mean,
                "pct_pval"      : float(pct_pval),
            }

# Save regional diff datasets
for metric in ["hist_climo", "hist_bias", "hist_bias_abs", "future_pct"]:
    ds = build_diff_dataset(results, metric, domains, models)
    if metric == "future_pct":
        ofilename = f"../highresmip_diagnostics/hrmip_nam_prec_{metric}_{fut_st_yr}-{fut_end_yr}.nc"
    else:
        ofilename = f"../highresmip_diagnostics/hrmip_nam_prec_{metric}_{hist_st_yr}-{hist_end_yr}.nc"
    if os.path.exists(ofilename):
        os.remove(ofilename)
    ds.to_netcdf(ofilename)

# Save p-values and boolean significance flags (p < 0.05) per resolution and domain
ds_pval = xr.Dataset()
for domain in domains:
    for res in resolutions:
        pvals = np.array([results[res][domain][m]["pct_pval"] for m in models])
        da    = xr.DataArray(pvals, coords={"model": models}, dims=["model"])
        ds_pval[f"pct_pval_{res}_{domain}"] = da
        ds_pval[f"pct_sig_{res}_{domain}"]  = da < 0.05
        
ofilename = f"../highresmip_diagnostics/hrmip_nam_prec_pct_pval_{fut_st_yr}-{fut_end_yr}.nc"
if os.path.exists(ofilename):
    os.remove(ofilename)
ds_pval.to_netcdf(ofilename)

print('Computed diagnostic values saved to netcdf.')

# Start Here on Re-Runs

In [ ]:
## +++ CACHE LOGIC +++ ##
# First run:  cells above (data loading → sig testing → regional analysis) must be run first;
#             this cell then writes cache files so subsequent runs are fast.
#
# Re-runs:    skip to here after running imports / helpers / models / domains / obs cells.

cache_dir   = '../highresmip_diagnostics/cache/'
cache_files = {
    (res, model): os.path.join(
        cache_dir,
        f"pr_future_{'lr' if res == 'lr' else 'hr'}_{lr_map[model] if res == 'lr' else hr_map[model]}.nc"
    )
    for res in resolutions for model in models
}
use_cache = all(os.path.exists(f) for f in cache_files.values())

if use_cache:
    print("Loading from cache...")
    future_diff      = {res: {} for res in resolutions}
    future_diff_mask = {res: {} for res in resolutions}
    jas_mean         = {res: {exp: {} for exp in experiments} for res in resolutions}
    obs_regrid       = {res: {} for res in resolutions}
    cfg_resolution   = {}

    for res in resolutions:
        for model in models:
            ds  = xr.open_dataset(cache_files[(res, model)])
            cfg = lr_map[model] if res == 'lr' else hr_map[model]
            future_diff[res][model]                    = ds['future_diff']
            future_diff_mask[res][model]               = ds['future_diff_mask']
            jas_mean[res]['highresSST-present'][model] = ds['jas_mean_present']
            jas_mean[res]['highresSST-future'][model]  = ds['jas_mean_future']
            obs_regrid[res][model]                     = ds['imerg_regrid']
            cfg_resolution[cfg]                        = int(ds.attrs['nlon'])
    print("Done.")

else:
    print("Writing cache for future runs...")
    os.makedirs(cache_dir, exist_ok=True)
    for res in resolutions:
        for model in models:
            cfg    = lr_map[model] if res == 'lr' else hr_map[model]
            mask = future_diff_mask[res][model]
            mask_da = xr.DataArray(
                            mask,
                            dims=future_diff[res][model].dims,
                            coords=future_diff[res][model].coords,
                        )

            ds_out = xr.Dataset({
                'future_diff':      future_diff[res][model],
                'future_diff_mask': mask_da,
                'jas_mean_present': jas_mean[res]['highresSST-present'][model],
                'jas_mean_future':  jas_mean[res]['highresSST-future'][model],
                'imerg_regrid':     obs_regrid[res][model],
            })
            ds_out.attrs['nlon'] = cfg_resolution[cfg]
            ds_out.to_netcdf(cache_files[(res, model)])
    print(f"Saved {len(resolutions) * len(models)} cache files to {cache_dir}")

#=== Load pre-saved regional summary datasets ===
future_pct    = xr.open_dataset(f'../highresmip_diagnostics/hrmip_nam_prec_future_pct_{fut_st_yr}-{fut_end_yr}.nc')
hist_bias     = xr.open_dataset(f'../highresmip_diagnostics/hrmip_nam_prec_hist_bias_{hist_st_yr}-{hist_end_yr}.nc')
hist_bias_abs = xr.open_dataset(f'../highresmip_diagnostics/hrmip_nam_prec_hist_bias_abs_{hist_st_yr}-{hist_end_yr}.nc')
pct_pval_ds   = xr.open_dataset(f'../highresmip_diagnostics/hrmip_nam_prec_pct_pval_{fut_st_yr}-{fut_end_yr}.nc')

# FIGURES

In [ ]:
def plot_map_grid(get_data, cmap, norm, cbar_label, cbar_ticks,
                  cbar_extend='both', caption=''):
    """Plot a 2 (lr/hr) × n_models grid of maps.

    Parameters
    ----------
    get_data : callable(res, model) → DataArray
    cbar_extend : {'both', 'max', 'min', 'neither'}
    caption : str, optional figure note placed below the axes
    """
    text_kw  = {'color': 'k', 'weight': 'bold', 'size': 14, 'ha': 'center', 'va': 'bottom'}
    text_kw2 = {'color': 'k', 'weight': 'bold', 'size': 20, 'ha': 'center', 'va': 'center'}
    text_kw3 = {'color': 'k', 'weight': 'normal', 'size': 14, 'ha': 'left',   'va': 'center'}
    letters  = list('ABCDEFGHIJKLMNOPQRSTUVWXYZ')
    tx, ty   = -110.5, 40.25
    map_tkw  = {'color': 'black', 'weight': 'normal', 'size': 10}
    trans = proj = ccrs.PlateCarree()
    map_bnds = [239, 260, 15, 40]

    fig, axes = plt.subplots(
        nrows=2, ncols=len(models), figsize=(25, 7),
        layout='constrained', subplot_kw={'projection': proj}
    )

    cf = None
    for i, res in enumerate(['lr', 'hr']):
        for j, model in enumerate(models):
            da = get_data(res, model)
            cf = axes[i, j].pcolormesh(da.lon, da.lat, da, cmap=cmap, norm=norm, transform=trans)
            if i == 0:
                axes[i, j].text(tx, ty, model, **text_kw)

    nrows, ncols = axes.shape
    for k, ax in enumerate(axes.flat):
        row = k // ncols
        col = k % ncols
        # add box around NAM sub-domains
        poly1=patches.Polygon(coords_map['north'], closed=True, ec='firebrick', fc='none', lw=1.5, ls='--', transform=ccrs.PlateCarree(), zorder=100)
        poly2=patches.Polygon(coords_map['south'], closed=True, ec='red', fc='none', lw=1.5, ls='--', transform=ccrs.PlateCarree(), zorder=100)
        ax.add_patch(poly1)
        ax.add_patch(poly2)
        ax.text(-121.5, ty + 1, letters[k], **text_kw2)
        ax.coastlines(color='k', linewidth=.75)
        ax.add_feature(cfeature.STATES,  edgecolor='k', linewidth=.75)
        ax.add_feature(cfeature.BORDERS, edgecolor='k', linewidth=.75)
        ax.set_extent(map_bnds, crs=trans)
        gl = ax.gridlines(crs=trans, lw=.5, colors='black', alpha=1.0,
                          linestyle='--', zorder=10, draw_labels=True)
        gl.bottom_labels = (row == nrows - 1)
        gl.left_labels   = (col == 0)
        gl.top_labels    = False
        gl.right_labels  = False
        gl.xformatter = LONGITUDE_FORMATTER
        gl.yformatter = LATITUDE_FORMATTER
        gl.xlabel_style = map_tkw
        gl.ylabel_style = map_tkw

    if caption:
        fig.text(0.025, -0.05, caption, **text_kw3)

    cax  = fig.add_axes([1.01, 0.1, 0.02, 0.8])
    cbar = fig.colorbar(cf, ticks=cbar_ticks, orientation='vertical',
                        extend=cbar_extend, cax=cax)
    cbar.set_label(cbar_label, labelpad=25, rotation=270, size=14,
                   fontweight='normal', ha='center')
    cbar.ax.tick_params(labelsize=14)
    for tick in cbar.ax.yaxis.get_major_ticks():
        tick.label2.set_fontweight('normal')

    return fig


def plot_panel_horizontal(ax, dat_lr, dat_hr, sig_lr=None, sig_hr=None):
    """Horizontal dot plot comparing LR and HR regional mean values per model.

    Parameters
    ----------
    dat_lr, dat_hr : xr.DataArray with 'model' dimension
    sig_lr, sig_hr : xr.DataArray (bool, 'model' dim), optional
        Whether each model's future change is statistically significant.
        Filled marker = significant; open marker = not significant.
        LR markers use black edge/face; HR markers use darkgoldenrod edge/face.
    """
    line_kw   = dict(color='0.7', lw=2, zorder=1)
    marker_kw = dict(marker='s', linewidth=1.5, zorder=3)

    yvals = np.arange(len(models))

    for y, model in zip(yvals, models):
        lr_cfg = lr_map[model]
        hr_cfg = hr_map[model]

        lr_val = dat_lr.sel(model=model)
        hr_val = dat_hr.sel(model=model)

        lr_size = 40000 / cfg_resolution[lr_cfg]
        hr_size = 40000 / cfg_resolution[hr_cfg]

        lr_sig = bool(sig_lr.sel(model=model)) if sig_lr is not None else True
        hr_sig = bool(sig_hr.sel(model=model)) if sig_hr is not None else True

        ax.scatter(lr_val, y, s=lr_size,
                   facecolor='black' if lr_sig else 'white',
                   edgecolor='k', **marker_kw)
        ax.scatter(hr_val, y, s=hr_size,
                   facecolor='darkgoldenrod' if hr_sig else 'white',
                   edgecolor='darkgoldenrod', **marker_kw)

        ax.plot([lr_val, hr_val], [y, y], **line_kw)

    ax.axvline(0, color='k', ls='--', lw=1)
    ax.set_yticks(yvals)
    ax.set_yticklabels(models)
    ax.invert_yaxis()
    ax.xaxis.tick_top()
    ax.tick_params(axis='x', bottom=False, labelbottom=False)


def plot_scatter_comparison(x_ds, y_ds, xkey_fn, ykey_fn, xlabel, ylabel, suptitle,
                             domains=None, titles=None, xlim=(-50, 50), ylim=(-50, 50),
                             label_offset=(0, -2), label_ha='center',
                             show_oneone=True, show_title=True,
                             sig_ds=None, xsigkey_fn=None, ysigkey_fn=None):
    """Two-panel (per domain) scatter plot.

    Parameters
    ----------
    x_ds, y_ds       : xr.Dataset — datasets containing x and y values
    xkey_fn, ykey_fn : callable(domain) → variable name in ds
    show_oneone      : bool — draw 1:1 diagonal line
    show_title       : bool — draw domain title inside each panel
    sig_ds           : xr.Dataset, optional — contains boolean significance flags
    xsigkey_fn       : callable(domain) → sig_ds variable for x-axis significance
    ysigkey_fn       : callable(domain) → sig_ds variable for y-axis significance
                       Marker fill: both sig → black, one sig → gray, neither → open
    """
    if domains is None:
        domains = ['north', 'south']
    if titles is None:
        titles  = ['NORTHERN DOMAIN', 'SOUTHERN DOMAIN']

    label_text_kw = dict(fontsize=8, ha=label_ha, va='top', clip_on=True, zorder=100)
    title_text_kw = dict(fontweight='bold', fontsize=12, ha='left', va='top')

    fig, axes = plt.subplots(2, 1, figsize=(5, 8), sharex=True, sharey=True,
                             layout='constrained')

    for ax, domain, title in zip(axes, domains, titles):
        xvals  = x_ds[xkey_fn(domain)].values
        yvals  = y_ds[ykey_fn(domain)].values
        labels = y_ds[ykey_fn(domain)].model.values

        if sig_ds is not None:
            xsig = sig_ds[xsigkey_fn(domain)].values.astype(bool) if xsigkey_fn else np.ones(len(xvals), dtype=bool)
            ysig = sig_ds[ysigkey_fn(domain)].values.astype(bool) if ysigkey_fn else np.ones(len(yvals), dtype=bool)
            for x, y, label, xs, ys in zip(xvals, yvals, labels, xsig, ysig):
                fc = 'k' if (xs and ys) else ('gray' if (xs or ys) else 'none')
                ax.scatter([x], [y], s=50, ec='k', fc=fc)
                ax.text(x + label_offset[0], y + label_offset[1], label, **label_text_kw)
        else:
            ax.scatter(xvals, yvals, s=50, ec='k', fc='k')
            for x, y, label in zip(xvals, yvals, labels):
                ax.text(x + label_offset[0], y + label_offset[1], label, **label_text_kw)

        if show_title:
            title_x = xlim[0] + 0.02 * (xlim[1] - xlim[0])
            title_y = ylim[1] - 0.02 * (ylim[1] - ylim[0])
            ax.text(title_x, title_y, title, **title_text_kw)

        ax.set(xlim=xlim, ylim=ylim)
        ax.axhline(0, color='k', ls='--', lw=.75)
        ax.axvline(0, color='k', ls='--', lw=.75)
        if show_oneone:
            pad = max(abs(xlim[0]), abs(ylim[0])) + 50
            ax.plot([-pad, pad], [-pad, pad], color='r', ls='-', lw=1, zorder=0)
        ax.set_ylabel(ylabel)

    axes[1].set_xlabel(xlabel)
    if suptitle:
        fig.suptitle(suptitle, fontsize=12)
    return fig

## Maps

### absolute and percent change in precipitation between high- and low-resolution future and historical sims

In [ ]:
# Future precip change — significance-masked
dcmap, _, _, _ = get_settings(field='precip', diff=True)
dnorm = mpl.colors.BoundaryNorm(np.linspace(-3, 3, 25), dcmap.N)

plot_map_grid(
    get_data   = lambda res, model: future_diff_mask[res][model],
    cmap       = dcmap,
    norm       = dnorm,
    cbar_label = '$\\Delta$Precipitation [mm/day]',
    cbar_ticks = np.arange(-3, 4, 1),
    caption    = (f'Future ({fut_st_yr}–{fut_end_yr}) $−$ Historical ({hist_st_yr}–{hist_end_yr}) JAS precipitation change. '
                  f'Masked for statistical significance. '
                  f'Dashed boxes: southern (light red) and northern (dark red) NAM domains.')
);
#plt.savefig(f'{opath}/highresmip_future_diff_masked.png', bbox_inches='tight')

In [ ]:
# Future precip change — unmasked
dcmap, _, _, _ = get_settings(field='precip', diff=True)
dnorm = mpl.colors.BoundaryNorm(np.linspace(-2, 2, 25), dcmap.N)

plot_map_grid(
    get_data   = lambda res, model: future_diff[res][model],
    cmap       = dcmap,
    norm       = dnorm,
    cbar_label = '$\\Delta$Precipitation [mm/day]',
    cbar_ticks = np.arange(-2, 3, 1),
    caption    = (f'Future ({fut_st_yr}–{fut_end_yr}) $−$ Historical ({hist_st_yr}–{hist_end_yr}) JAS precipitation change. '
                  'Not masked for statistical significance. '
                  'Dashed boxes: southern (light red) and northern (dark red) NAM domains.')
);
#plt.savefig(f'{opath}/highresmip_future_diff_unmasked.png', bbox_inches='tight')

In [ ]:
# Future precip change as % of historical rate
dcmap, _, _, _ = get_settings(field='precip', diff=True)
dnorm = mpl.colors.BoundaryNorm(np.linspace(-100, 100, 21), dcmap.N)

plot_map_grid(
    get_data   = lambda res, model: (
        future_diff[res][model] / jas_mean[res]['highresSST-present'][model] * 100),
    cmap       = dcmap,
    norm       = dnorm,
    cbar_label = '$\\Delta$Daily Precipitation Rate [%]',
    cbar_ticks = np.linspace(-100, 100, 11),
    caption    = (f'Change in relative daily precipitation rate by {fut_st_yr}–{fut_end_yr} due to SSP5-8.5 GHG forcing: '
                  '(future$−$hist) / hist. Not masked for statistical significance.')
);
#plt.savefig(f'{opath}/highresmip_future_pct_change.png', bbox_inches='tight')

### historical and future precipitation climatologies

In [ ]:
# Historical JAS climatology
pcmap, _, _, _ = get_settings(field='precip', diff=False)
pnorm = mpl.colors.BoundaryNorm(np.linspace(0, 12, 25), pcmap.N)

plot_map_grid(
    get_data     = lambda res, model: jas_mean[res]['highresSST-present'][model],
    cmap         = pcmap,
    norm         = pnorm,
    cbar_label   = 'Precipitation Rate [mm/day]',
    cbar_ticks   = np.arange(0, 13, 1),
    cbar_extend  = 'max',
    caption      = ('Mean JAS daily precipitation rate — historical period (1950–2014). '
                    'Dashed boxes: southern (light red) and northern (dark red) NAM domains.')
)
#plt.savefig(f'{opath}/highresmip_hist_climo.png', bbox_inches='tight')

In [ ]:
# Future JAS climatology
pcmap, _, _, _ = get_settings(field='precip', diff=False)
pnorm = mpl.colors.BoundaryNorm(np.linspace(0, 12, 25), pcmap.N)

plot_map_grid(
    get_data     = lambda res, model: jas_mean[res]['highresSST-future'][model],
    cmap         = pcmap,
    norm         = pnorm,
    cbar_label   = 'Precipitation Rate [mm/day]',
    cbar_ticks   = np.arange(0, 13, 1),
    cbar_extend  = 'max',
    caption      = ('Mean JAS daily precipitation rate — future period (2040–2050), SSP5-8.5 GHG forcing. '
                    'Dashed boxes: southern (light red) and northern (dark red) NAM domains.')
)
#plt.savefig(f'{opath}/highresmip_future_climo.png', bbox_inches='tight')

## Comparison of regional mean biases in high- and low-resolution configurations

In [ ]:
# === Combined 4-panel comparison ===
#   Columns = NORTHERN / SOUTHERN domain
#   Row 1 (top)    : per-model LR vs HR future % change (dot plot)         -> A (north), B (south)
#   Row 2 (bottom) : HR-LR hist bias diff vs HR-LR future %-change diff (scatter) -> C, D
#   Panels share the Y-axis within each row (same y-scale); x-axes are independent.

# --- Settings ---
arrow_text_kw = {'size': 10, 'weight': 'bold', 'color': 'k', 'ha': 'center', 'va': 'center'}
col_title_kw  = {'size': 16, 'weight': 'bold', 'color': 'black'}
legend_kw = {'loc': 'center left', 'bbox_to_anchor': (.99, 0.5), 'ncol': 1, 'fontsize': 10,
               'labelcolor': 'k', 'frameon': False}
legend_handles = [
    Line2D([0], [0], marker='s', color='w', markeredgecolor='k',
           markerfacecolor='k', markersize=10, label='lower res.'),
    Line2D([0], [0], marker='s', color='w', markeredgecolor='darkgoldenrod',
           markerfacecolor='darkgoldenrod', markersize=5, label='higher res.'),
    Line2D([0], [0], marker='s', color='w', markeredgecolor='0.3',
           markerfacecolor='0.3', markersize=7, label='$p$ < 0.1'),
    Line2D([0], [0], marker='s', color='w', markeredgecolor='0.3',
           markerfacecolor='none', markersize=7, label='$p$ ≥ 0.1'),
]

# Scatter legend: marker fill encodes which of LR / HR future-change is significant
scatter_legend_handles = [
    Line2D([0], [0], marker='o', color='w', markeredgecolor='k',
           markerfacecolor='k',    markersize=8, label='LR & HR $p$<0.1'),
    Line2D([0], [0], marker='o', color='w', markeredgecolor='k',
           markerfacecolor='gray', markersize=8, label='LR or HR $p$<0.1'),
    Line2D([0], [0], marker='o', color='w', markeredgecolor='k',
           markerfacecolor='none', markersize=8, label='neither sig'),
]
scatter_legend_kw = {'loc': 'center left', 'bbox_to_anchor': (.99, 0.5), 'ncol': 1,
                       'fontsize': 9, 'labelcolor': 'k', 'frameon': False}

panel_titles = ['NORTHERN DOMAIN', 'SOUTHERN DOMAIN']

# Dot-plot (left/top) x-axis label
dot_xlabel = (f'$\Delta$Relative Precipitation [%]')

# Scatter (right/bottom) settings
scatter_xlim   = (-1.5, 0.5)
scatter_ylim   = (-45, 45)
scatter_xlabel = ('High-Res.$-$Low-Res.\n'
                  f'Difference in Absolute Historical ({hist_st_yr}-{hist_end_yr})\n'
                  'Precipitation Bias (mm/day)')
scatter_ylabel = ('High-Res.$-$Low-Res.\n'
                  f'Difference of Projected ({fut_st_yr}-{fut_end_yr})\n'
                  'Change in Relative Precipitation (%)')
label_offset   = (-0.025, -0.1)
label_text_kw  = dict(fontsize=8, ha='right', va='top', clip_on=True, zorder=100)

# --- Sub-panel letters + axis-description arrows (positions are tunable knobs) ---
panel_letter_kw = dict(fontsize=22, fontweight='bold', va='top', ha='left', zorder=200)
axis_arrow_kw   = dict(arrowstyle='<->', color='k', lw=1.5)
# vertical wetting/drying arrow, left of the (shared) scatter y-axis label
Y_ARROW_X, Y_EFF_X, Y_TOP, Y_BOT = -0.3, -0.35, 0.92, 0.08
# horizontal less/more-bias arrow, beneath panel C (below its 3-line x-axis label)
X_ARROW_Y, X_EFF_DY = -0.25, -0.025

def add_yaxis_arrow(ax):
    ax.annotate('', xy=(Y_ARROW_X, Y_TOP), xytext=(Y_ARROW_X, Y_BOT),
                xycoords='axes fraction', arrowprops=axis_arrow_kw, annotation_clip=False)
    ax.text(Y_ARROW_X, Y_TOP + 0.03, 'Wetting', transform=ax.transAxes, clip_on=False, **arrow_text_kw)
    ax.text(Y_ARROW_X, Y_BOT - 0.03, 'Drying', transform=ax.transAxes, clip_on=False, **arrow_text_kw)
    ax.text(Y_EFF_X, 0.5, 'Increased Resolution Impact', transform=ax.transAxes,
            rotation=90, ha='center', va='center', fontsize=10, fontweight='bold', clip_on=False)

def add_xaxis_arrow(ax):
    ax.annotate('', xy=(0.93, X_ARROW_Y), xytext=(0.07, X_ARROW_Y),
                xycoords='axes fraction', arrowprops=axis_arrow_kw, annotation_clip=False)
    ax.text(0.02, X_ARROW_Y, 'Lower\nBias', transform=ax.transAxes, clip_on=False, **arrow_text_kw)
    ax.text(0.98, X_ARROW_Y, 'Higher\nBias', transform=ax.transAxes, clip_on=False, **arrow_text_kw)
    ax.text(0.5, X_ARROW_Y + X_EFF_DY, 'Increased Resolution Impact',
            transform=ax.transAxes, ha='center', va='top', fontsize=10, fontweight='bold', clip_on=False)

# --- Layout: rows = plot type, columns = domain; share Y within each row ---
fig, axes = plt.subplots(2, 2, figsize=(12, 10), sharey='row', constrained_layout=True)

dot_letters = ['A', 'B']
sc_letters  = ['C', 'D']

for col, (domain, title) in enumerate(zip(domains, panel_titles)):

    # ----- Top row: LR vs HR future % change (per-model dot plot) -----
    ax_dot = axes[0, col]
    dat_lr = future_pct[f'future_pct_lr_{domain}']
    dat_hr = future_pct[f'future_pct_hr_{domain}']
    sig_lr = pct_pval_ds[f'pct_sig_lr_{domain}']
    sig_hr = pct_pval_ds[f'pct_sig_hr_{domain}']

    plot_panel_horizontal(ax_dot, dat_lr, dat_hr, sig_lr=sig_lr, sig_hr=sig_hr)
    ax_dot.set(xlim=[-80, 85])
    ax_dot.set_title(title, pad=20, **col_title_kw)
    ax_dot.set_xlabel(dot_xlabel)
    ax_dot.xaxis.set_label_position('top')
    ax_dot.tick_params(axis='x', top=True, labeltop=True, bottom=False, labelbottom=False)
    ax_dot.text(0.04, 0.96, dot_letters[col], transform=ax_dot.transAxes, **panel_letter_kw)
    if col == 0:
        pass
    else:
        ax_dot.tick_params(labelleft=False)
        ax_dot.legend(handles=legend_handles, **legend_kw)   # right of panel B

    # ----- Bottom row: HR-LR bias-diff vs HR-LR future-%-change-diff scatter -----
    ax_sc = axes[1, col]
    xvals  = hist_bias_abs[f'hist_bias_abs_diff_{domain}'].values
    yvals  = future_pct[f'future_pct_diff_{domain}'].values
    labels = future_pct[f'future_pct_diff_{domain}'].model.values
    xsig   = pct_pval_ds[f'pct_sig_lr_{domain}'].values.astype(bool)
    ysig   = pct_pval_ds[f'pct_sig_hr_{domain}'].values.astype(bool)

    for x, y, label, xs, ys in zip(xvals, yvals, labels, xsig, ysig):
        fc = 'k' if (xs and ys) else ('gray' if (xs or ys) else 'none')
        ax_sc.scatter([x], [y], s=50, ec='k', fc=fc, clip_on=False)
        ax_sc.text(x + label_offset[0], y + label_offset[1], label, **label_text_kw)

    ax_sc.set(xlim=[-1.5, 0.5], ylim=[-90, 50])
    ax_sc.axhline(0, color='k', ls='--', lw=.75)
    ax_sc.axvline(0, color='k', ls='--', lw=.75)
    ax_sc.set_xlabel(scatter_xlabel)
    ax_sc.text(0.04, 0.96, sc_letters[col], transform=ax_sc.transAxes, **panel_letter_kw)
    if col == 0:
        ax_sc.set_ylabel(scatter_ylabel)
        add_yaxis_arrow(ax_sc)   # vertical wetting/drying arrow, left of the shared y-axis
        add_xaxis_arrow(ax_sc)   # horizontal less/more-bias arrow, beneath panel C
    else:
        ax_sc.tick_params(labelleft=False)
        add_xaxis_arrow(ax_sc)
        ax_sc.legend(handles=scatter_legend_handles, **scatter_legend_kw)

plt.savefig(f'../figs/prec_hrmip_future_delta_{hist_st_yr}-{hist_end_yr}_vs_{fut_st_yr}-{fut_end_yr}.png', transparent=False, bbox_inches='tight')
plt.savefig(f'../figs/prec_hrmip_future_delta_{hist_st_yr}-{hist_end_yr}_vs_{fut_st_yr}-{fut_end_yr}.pdf', transparent=False, bbox_inches='tight')


In [ ]:
# Historical bias: Low-Res. vs High-Res.
plot_scatter_comparison(
    x_ds         = hist_bias,
    y_ds         = hist_bias,
    xkey_fn      = lambda d: f'hist_bias_lr_{d}',
    ykey_fn      = lambda d: f'hist_bias_hr_{d}',
    sig_ds     = pct_pval_ds,
    xsigkey_fn = lambda d: f'pct_sig_lr_{d}',
    ysigkey_fn = lambda d: f'pct_sig_hr_{d}',
    xlabel       = 'Low-Res.',
    ylabel       = 'High-Res.',
    suptitle     = 'Historical Prec Bias (mm/day)',
    xlim         = (-2, 0.5),
    ylim         = (-2, 0.5),
    label_offset = (-0.02, -0.02),
    label_ha     = 'right',
);